In [ ]:
from logger import log_loss_to_csv
import torch
import torchvision
import os
from torch.utils.data import DataLoader, random_split, Subset
from tqdm import tqdm
from ldm.models.autoencoder import AutoencoderKL
from dataset import SimpleHandsDataset

# ==========================================
# 1. CONFIGURATION & DOSSIERS
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
config = {
    "batch_size": 4,
    "lr": 1e-5,
    "num_epochs": 5,
    "val_split": 0.2,
    "max_samples": 2000, # Limité à 20 pour ton test rapide
    "save_dir": "./result_vae",
    "model_name": "vae_hands_latest.pt"
}

os.makedirs(config["save_dir"], exist_ok=True)

# ==========================================
# 2. PRÉPARATION DES DONNÉES
# ==========================================
full_dataset = SimpleHandsDataset(root_dir="./data/hands")

# Réduction de la taille du dataset si nécessaire
if config["max_samples"] < len(full_dataset):
    indices = torch.randperm(len(full_dataset))[:config["max_samples"]]
    dataset = Subset(full_dataset, indices)
else:
    dataset = full_dataset

train_size = int((1 - config["val_split"]) * len(dataset))
val_size = len(dataset) - train_size
train_ds, val_ds = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=config["batch_size"], shuffle=True)
val_loader = DataLoader(val_ds, batch_size=config["batch_size"], shuffle=False)

print(f"Dataset : {train_size} images (train) | {val_size} images (val)")

# ==========================================
# 3. MODÈLE ET OPTIMISEUR
# ==========================================
model = AutoencoderKL(
    ddconfig={
        "double_z": True, "z_channels": 4, "resolution": 256,
        "in_channels": 3, "out_ch": 3, "ch": 128,
        "ch_mult": [1, 2, 4, 4], "num_res_blocks": 2, "attn_resolutions": []
    },
    lossconfig={
        "target": "ldm.modules.losses.LPIPSWithDiscriminator",
        "params": {"disc_start": 50001, "kl_weight": 0.000001, "disc_weight": 0.5}
    },
    embed_dim=4
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"])

# ==========================================
# 4. BOUCLE D'ENTRAÎNEMENT
# ==========================================
for epoch in range(config["num_epochs"]):
    model.train()
    epoch_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config['num_epochs']}")
    
    for batch in pbar:
        if isinstance(batch, list): 
            batch = batch[0] # On ne garde que l'image
            
        batch = batch.to(device)
        
        # Forward pass
        reconstructions, posterior = model(batch)
        
        # 1. Calcul de la perte de reconstruction (MSE)
        recon_loss = torch.nn.functional.mse_loss(reconstructions, batch)
        
        # 2. Calcul de la perte de divergence KL
        kl_loss = posterior.kl().mean()
        
        # 3. Perte totale (On multiplie le KL par le kl_weight de ta config : 0.000001)
        loss = recon_loss + 0.000001 * kl_loss
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        pbar.set_postfix({
            "loss": f"{loss.item():.4f}", 
            "recon": f"{recon_loss.item():.4f}", 
            "kl": f"{kl_loss.item():.6f}"
        })

    # ==========================================
    # 5. PHASE DE VALIDATION & VISUALISATION
    # ==========================================
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for val_batch in val_loader:
            val_batch = val_batch.to(device)
            recons, _ = model(val_batch)
            
            v_loss = torch.nn.functional.mse_loss(recons, val_batch)
            val_loss += v_loss.item()
            
        avg_val = val_loss / len(val_loader)
        print(f" --- Validation Loss: {avg_val:.6f} ---")

        csv_file_path = os.path.join("./result_vae", "training_val_losses.csv")
        log_loss_to_csv(csv_file_path, epoch + 1, loss.item(), avg_val)
        
        # Sauvegarde d'un échantillon (Image originale vs Reconstruction)
        # On crée une grille pour comparer
        comparison = torch.cat([val_batch[:1], recons[:1]]) 
        save_path = os.path.join(config["save_dir"], f"reconst_epoch_{epoch+1}.png")
        torchvision.utils.save_image(comparison, save_path, normalize=True, nrow=2)

    # Sauvegarde du checkpoint
    torch.save(model.state_dict(), config["model_name"])

print("Entraînement du VAE terminé !")

Dataset : 1600 images (train) | 400 images (val)
making attention of type 'vanilla' with 512 in_channels
Working with z of shape (1, 4, 32, 32) = 4096 dimensions.
making attention of type 'vanilla' with 512 in_channels


C:\Users\thiba\.conda\envs\crossvit_cy\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\thiba\.conda\envs\crossvit_cy\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
c:\users\thiba\src\taming-transformers\taming\modules\losses\lpips.py:29: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pyt

loaded pretrained LPIPS loss from taming/modules/autoencoder/lpips\vgg.pth


Epoch 1/5: 100%|████████████████████████████████████████████████████| 400/400 [1:27:49<00:00, 13.17s/it, loss=0.007966]


 --- Validation Loss: 0.006981 ---


...

 --- Validation Loss: 0.003367 ---


Epoch 4/5: 100%|████████████████████████████████████████████████████| 400/400 [1:27:44<00:00, 13.16s/it, loss=0.003643]


 --- Validation Loss: 0.002922 ---


Epoch 5/5: 100%|████████████████████████████████████████████████████| 400/400 [1:27:44<00:00, 13.16s/it, loss=0.002478]


 --- Validation Loss: 0.002602 ---
Entraînement du VAE terminé !
